# Continuous Training on Kaggle - AFRICA GIANTS

This notebook trains the **Afrique Llama 8B** model on scraped and synthetic Tanzanian business/regulatory data.
It is programmatically triggered, runs SFT using QLoRA, runs evaluation, and uploads weights back to Hugging Face Hub.

In [ ]:
# Install dependencies required on Kaggle
!pip install -q -U transformers peft accelerate datasets bitsandbytes trl huggingface_hub

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import HfApi, create_repo, login, whoami

In [ ]:
# Login to Hugging Face using the token stored in Kaggle Secrets.
# Kaggle secret label must be exactly: AFRICA_GIANTS
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("AFRICA_GIANTS")
login(token=hf_token)

hf_user = whoami(token=hf_token)["name"]
print(f"Logged in to Hugging Face as: {hf_user}")

api_key = ""
try:
    api_key = user_secrets.get_secret("OPENAI_API_KEY")
except Exception:
    pass

In [ ]:
# Hugging Face repositories
BASE_MODEL = "prospAprospA007/Afrique-llama-8B"
ADAPTER_REPO = "prospAprospA007/africa-giants-adapter-v1"
MERGED_MODEL_REPO = "prospAprospA007/africa-giants-model-v1"
DATASET_REPO = "prospAprospA007/africa-giants-dataset"

# Backward-compatible aliases used by older cells
dataset_name = DATASET_REPO
base_model_name = BASE_MODEL
target_repo = ADAPTER_REPO

# Make sure target repos exist before training starts.
for repo_id, repo_type in [
    (ADAPTER_REPO, "model"),
    (MERGED_MODEL_REPO, "model"),
    (DATASET_REPO, "dataset"),
]:
    create_repo(repo_id=repo_id, repo_type=repo_type, private=True, exist_ok=True, token=hf_token)
    print(f"Ready: {repo_type} repo {repo_id}")

print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name, token=hf_token)
print(dataset)

In [ ]:
# Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer and model
print(f"Loading tokenizer and model for base model {base_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=hf_token, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Fallback to standard open model if private model is not accessible during testing
try:
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
        trust_remote_code=True
    )
except Exception as e:
    print(f"Failed to load {base_model_name}: {e}. Falling back to default Llama-3 8B.")
    fallback_model = "meta-llama/Meta-Llama-3-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(fallback_model, token=hf_token)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        fallback_model,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token
)

In [ ]:
# Prepare model for LoRA training
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Define SFT formatter
def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['instruction'])):
        inst = example['instruction'][i]
        inp = example['input'][i] if 'input' in example and example['input'][i] else ''
        out = example['output'][i]
        if inp:
            text = f"<|im_start|>system\nYou are a helpful assistant for Tanzanian business. Use the context to answer.<|im_end|>\n<|im_start|>user\nContext: {inp}\nQuestion: {inst}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        else:
            text = f"<|im_start|>system\nYou are a helpful assistant for Tanzanian business.<|im_end|>\n<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        output_texts.append(text)
    return output_texts

In [ ]:
# Set this to True for a tiny smoke test before full training.
SMOKE_TEST_TRAINING = True

# Set up SFTTrainer
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1 if SMOKE_TEST_TRAINING else 4,
    gradient_accumulation_steps=1 if SMOKE_TEST_TRAINING else 4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1 if SMOKE_TEST_TRAINING else 3,
    max_steps=10 if SMOKE_TEST_TRAINING else -1,
    bf16=True,
    optim="adamw_torch",
    save_strategy="epoch",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset.get("validation", None),
    peft_config=peft_config,
    max_seq_length=1024 if SMOKE_TEST_TRAINING else 2048,
    formatting_func=formatting_prompts_func,
    args=training_args,
    packing=False
)

print("Starting training...")
trainer.train()

In [ ]:
# Evaluate and check validation loss gate
eval_results = trainer.evaluate()
validation_loss = eval_results.get("eval_loss", 999.0)
print(f"Validation Loss: {validation_loss}")

# If validation loss is below threshold, push to registry
LOSS_THRESHOLD = 2.2
if validation_loss <= LOSS_THRESHOLD:
    print(f"Validation check passed! Pushing LoRA adapter to {ADAPTER_REPO}...")
    trainer.model.push_to_hub(ADAPTER_REPO, token=hf_token)
    tokenizer.push_to_hub(ADAPTER_REPO, token=hf_token)
    print("Adapter pushed successfully.")
else:
    print(f"Validation check failed. Loss {validation_loss} > threshold {LOSS_THRESHOLD}. Model deployment aborted.")

In [ ]:
# Optional: merge the LoRA adapter into the base model and push the merged model.
# Keep this False for the first Kaggle smoke test. Turn it on after the adapter push works.
MERGE_AND_PUSH_FULL_MODEL = False

if MERGE_AND_PUSH_FULL_MODEL and validation_loss <= LOSS_THRESHOLD:
    print(f"Merging adapter {ADAPTER_REPO} into base model {BASE_MODEL}...")
    base_for_merge = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        token=hf_token,
        trust_remote_code=True,
    )
    merged_model = PeftModel.from_pretrained(base_for_merge, ADAPTER_REPO, token=hf_token)
    merged_model = merged_model.merge_and_unload()
    print(f"Pushing merged model to {MERGED_MODEL_REPO}...")
    merged_model.push_to_hub(MERGED_MODEL_REPO, token=hf_token)
    tokenizer.push_to_hub(MERGED_MODEL_REPO, token=hf_token)
    print("Merged model pushed successfully.")
else:
    print("Skipping merged model push. Adapter repo is the deployment target for now.")